# HDiv-VIM hex (RT1) vs collocation-MMMM hex -- cross-validation

Two INDEPENDENT hex soft-iron demag backends solved on the **same structured hex mesh** and compared, on a
**cube** and a **C-yoke** (non-convex, reentrant, non-uniform M):

| backend | element | how it is driven here |
|---|---|---|
| **collocation MMMM** | six-face surface charge (Yano element, 6 DOF/hex) | public `rad.Solve(..., demag_backend="collocation_mmmm")` on the `ObjHexahedron` iron from `radia.vim.soft_iron_from_mesh` |
| **HDiv-VIM RT1** | H(div) RT1 flux -> charge Gram `N = B^T G B` | the wired hex Gram `build_charge_gram(HDiv(hexmesh, order=1))` + the shipped `_solve_linear_mass_riesz_cpp` |

The two are compared **order-independently** by (a) the volume-average magnetization and (b) the **iron's
external B field** at probe points (the same `rad.Fld` kernel for both) -- the engineering quantity
(CLAUDE.md field comparison = vector difference).  Neither backend is ground truth; agreement + a shrinking
gap under refinement is the cross-validation evidence.  The cube also anchors HDiv on the **exact** demag
factor 1/3.


In [1]:
import os, sys, json, platform
import numpy as np
import ngsolve as ng
import radia as rad

# the topic helper (docs/hdiv_vim/hex_vs_mmmm_helpers.py) -- nbconvert runs with cwd = the notebook dir
sys.path.insert(0, os.getcwd())
import hex_vs_mmmm_helpers as HX

RESULTS = {
    "radia_version": rad.__version__,
    "numpy_version": np.__version__,
    "platform": platform.platform(),
    "note": ("HDiv-VIM hex RT1 (wired charge Gram + shipped mass-Riesz CG) vs collocation-MMMM hex "
             "(rad.Solve demag_backend=collocation_mmmm) on the SAME hex mesh; metrics = volume-avg M + "
             "iron external B via rad.Fld."),
}
print("radia", rad.__version__, "| numpy", np.__version__, "|", platform.platform())


radia 4.95.5 | numpy 2.4.6 | Windows-2022Server-10.0.20348-SP0


## Architectural note (why the HDiv-hex path looks 'internal')

The hex RT1 charge Gram is wired at `build_charge_gram(HDiv(hexmesh, order=1))`, but the public entry
`radia.vim.hdiv_demag_solve` **deliberately guards non-tet** (production `auto` routes a hex iron to the
collocation MMMM backend -- see `radia.set_demag_backend`).  So to exercise **HDiv-VIM on hex** we drive the
wired hex Gram with the **shipped** production linear solver `_solve_linear_mass_riesz_cpp` -- exactly the
symmetric mass-Riesz CG that the tet `_solve_highorder` uses, only the Gram is the hex one.  This is the real
HDiv-hex solve; flipping the public entry to expose hex is a pending policy decision (KEEP-BOTH: MMMM stays
the coarse tier).  All of that lives in `hex_vs_mmmm_helpers.py`.


## Cube (uniform +z field, mu_r = 100)

Same `MakeStructured3DMesh` hex cube for both backends; refine n = 4, 6, 8, 10.

In [2]:
MU_R = 100.0
H0 = 1000.0
PROBE_CUBE = [[0, 0, 0.015], [0, 0, 0.02], [0, 0, 0.03], [0.008, 0, 0.015], [0.006, 0.006, 0.014]]

cube_rows = []
print(f"{'n':>2} {'nhex':>5} {'HDiv Mz':>9} {'MMMM Mz':>9} {'dMz':>8} {'demag':>7} {'it':>3} {'dB_ext_max':>10}")
for n in (4, 6, 8, 10):
    with ng.TaskManager():
        hd = HX.hdiv_hex_solve(HX.cube_mesh(n), MU_R, (0, 0, H0))
    with ng.TaskManager():
        mm = HX.mmmm_hex_solve(HX.cube_mesh(n), MU_R, (0, 0, H0), PROBE_CUBE)
    with ng.TaskManager():
        hdB = HX.iron_external_field(HX.cube_mesh(n), hd["M"], PROBE_CUBE)
    dM, dB, dBmax = HX.agreement(hd["M_avg"], mm["M_avg"], hdB, mm["B_iron"], 2)
    row = dict(n=n, n_hex=hd["n_el"], hdiv_Mz=float(hd["M_avg"][2]), mmmm_Mz=float(mm["M_avg"][2]),
               dMz_rel=dM, hdiv_demag=hd["demag"], hdiv_iters=hd["iters"], dB_ext_max=dBmax)
    cube_rows.append(row)
    print(f"{n:>2} {hd['n_el']:>5} {row['hdiv_Mz']:>9.1f} {row['mmmm_Mz']:>9.1f} {100*dM:>7.3f}% "
          f"{hd['demag']:>7.4f} {hd['iters']:>3} {100*dBmax:>9.3f}%")
RESULTS["cube"] = cube_rows


 n  nhex   HDiv Mz   MMMM Mz      dMz   demag  it dB_ext_max


[WARNING] Imported 64 hexahedral elements. Hexahedra may cause numerical issues in Radia MMM.
[WARNING] Imported 64 hexahedral elements. Hexahedra may cause numerical issues in Radia MMM.
 4    64    3481.1    3423.9   1.673%  0.3328  23     9.140%


[WARNING] Imported 216 hexahedral elements. Hexahedra may cause numerical issues in Radia MMM.
[WARNING] Imported 216 hexahedral elements. Hexahedra may cause numerical issues in Radia MMM.
 6   216    3487.9    3454.3   0.970%  0.3330  23     4.407%


[WARNING] Imported 512 hexahedral elements. Hexahedra may cause numerical issues in Radia MMM.


[WARNING] Imported 512 hexahedral elements. Hexahedra may cause numerical issues in Radia MMM.
 8   512    3490.9    3467.7   0.669%  0.3331  23     2.888%


[WARNING] Imported 1000 hexahedral elements. Hexahedra may cause numerical issues in Radia MMM.


[WARNING] Imported 1000 hexahedral elements. Hexahedra may cause numerical issues in Radia MMM.
10  1000    3492.6    3475.1   0.503%  0.3331  23     1.946%


## C-yoke (voxelized hex, in-plane +y field, mu_r = 100)

The current-tree C-yoke is a tet OCC body; to compare the two **hex** backends we voxelize the **same**
`cyoke()` shape (outer box minus inner cavity minus the +x gap opening) into a structured hex mesh.  A
genuine non-convex, reentrant-corner, non-uniform-M body -- the hard demag case.  Refine h = 8, 6 mm.

In [3]:
AXIS = 1  # applied field +y (in the C plane -> drives flux round the C)
PROBE_CT = [[0, 0, 0], [-0.025, 0, 0], [0.045, 0, 0], [0.1, 0, 0], [0, 0.1, 0], [0, 0, 0.05]]

def hvec(v):
    a = [0.0, 0.0, 0.0]; a[AXIS] = v; return tuple(a)

ct_rows = []
print(f"{'h':>6} {'nhex':>5} {'HDiv My':>9} {'MMMM My':>9} {'dMy':>8} {'demag':>7} {'it':>3} {'dB_ext_max':>10}")
for h in (0.008, 0.006):
    with ng.TaskManager():
        mesh, nh = HX.cyoke_mesh(h); hd = HX.hdiv_hex_solve(mesh, MU_R, hvec(H0))
    with ng.TaskManager():
        mesh2, _ = HX.cyoke_mesh(h); mm = HX.mmmm_hex_solve(mesh2, MU_R, hvec(H0), PROBE_CT)
    with ng.TaskManager():
        mesh3, _ = HX.cyoke_mesh(h); hdB = HX.iron_external_field(mesh3, hd["M"], PROBE_CT)
    dM, dB, dBmax = HX.agreement(hd["M_avg"], mm["M_avg"], hdB, mm["B_iron"], AXIS)
    row = dict(h=h, n_hex=hd["n_el"], hdiv_My=float(hd["M_avg"][AXIS]), mmmm_My=float(mm["M_avg"][AXIS]),
               dMy_rel=dM, hdiv_demag=hd["demag"], hdiv_iters=hd["iters"], dB_ext_max=dBmax)
    ct_rows.append(row)
    print(f"{h:>6.3f} {hd['n_el']:>5} {row['hdiv_My']:>9.1f} {row['mmmm_My']:>9.1f} {100*dM:>7.3f}% "
          f"{hd['demag']:>7.4f} {hd['iters']:>3} {100*dBmax:>9.3f}%")
RESULTS["ctype"] = ct_rows


     h  nhex   HDiv My   MMMM My      dMy   demag  it dB_ext_max


[WARNING] Imported 435 hexahedral elements. Hexahedra may cause numerical issues in Radia MMM.
[WARNING] Imported 435 hexahedral elements. Hexahedra may cause numerical issues in Radia MMM.


 0.008   435    9582.6    9527.5   0.578%  0.3154  34     2.907%


[WARNING] Imported 1064 hexahedral elements. Hexahedra may cause numerical issues in Radia MMM.


[WARNING] Imported 1064 hexahedral elements. Hexahedra may cause numerical issues in Radia MMM.
 0.006  1064    9444.3    9406.7   0.400%  0.3156  34     1.583%


## Convergence

Both metrics -- the relative difference of the volume-average M and the max relative external-B vector
difference -- shrink monotonically as the hex mesh refines: the two independent discretizations approach the
same continuum solution.

In [4]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(8.4, 3.3))
cn = [r["n_hex"] for r in cube_rows]
ax[0].loglog(cn, [100 * r["dMz_rel"] for r in cube_rows], "o-", label="rel. diff M (avg)")
ax[0].loglog(cn, [100 * r["dB_ext_max"] for r in cube_rows], "s--", label="rel. diff B_ext (max)")
ax[0].set_xlabel("hex count"); ax[0].set_ylabel("HDiv vs MMMM  [%]")
ax[0].grid(True, which="both", ls=":"); ax[0].legend(fontsize=8); ax[0].text(0.05, 0.05, "cube", transform=ax[0].transAxes)
tn = [r["n_hex"] for r in ct_rows]
ax[1].loglog(tn, [100 * r["dMy_rel"] for r in ct_rows], "o-", label="rel. diff M (avg)")
ax[1].loglog(tn, [100 * r["dB_ext_max"] for r in ct_rows], "s--", label="rel. diff B_ext (max)")
ax[1].set_xlabel("hex count"); ax[1].set_ylabel("HDiv vs MMMM  [%]")
ax[1].grid(True, which="both", ls=":"); ax[1].legend(fontsize=8); ax[1].text(0.05, 0.05, "C-yoke", transform=ax[1].transAxes)
fig.tight_layout()
fig


<Figure size 840x330 with 2 Axes>

## Conclusion

- **HDiv-VIM hex (RT1) and collocation-MMMM hex agree to sub-% on the volume-average M** at moderate
  resolution, on BOTH the cube and the non-convex C-yoke, and the gap (M and external B) **shrinks
  monotonically** under refinement -- the signature of two consistent discretizations of the same operator.
- The cube pins HDiv on the **exact** demag factor 1/3 (0.333); the HDiv +N mass-Riesz CG iteration count is
  **mesh-robust** (bounded, ~constant).
- Both are KEPT (CLAUDE.md): HDiv-VIM = loop-free per-element accuracy (primary), collocation MMMM = the fast
  coarse tier.  This notebook is the Python-level cross-check that they land on the same field.


In [5]:
print(json.dumps(RESULTS, indent=1))

{
 "radia_version": "4.95.5",
 "numpy_version": "2.4.6",
 "platform": "Windows-2022Server-10.0.20348-SP0",
 "note": "HDiv-VIM hex RT1 (wired charge Gram + shipped mass-Riesz CG) vs collocation-MMMM hex (rad.Solve demag_backend=collocation_mmmm) on the SAME hex mesh; metrics = volume-avg M + iron external B via rad.Fld.",
 "cube": [
  {
   "n": 4,
   "n_hex": 64,
   "hdiv_Mz": 3481.1195264000935,
   "mmmm_Mz": 3423.851955413273,
   "dMz_rel": 0.016726065184061948,
   "hdiv_demag": 0.33284089469202377,
   "hdiv_iters": 23,
   "dB_ext_max": 0.09139702579348845
  },
  {
   "n": 6,
   "n_hex": 216,
   "hdiv_Mz": 3487.8683060542276,
   "mmmm_Mz": 3454.3443109167865,
   "dMz_rel": 0.009704879456137287,
   "hdiv_demag": 0.3329866776436565,
   "hdiv_iters": 23,
   "dB_ext_max": 0.044065314691909434
  },
  {
   "n": 8,
   "n_hex": 512,
   "hdiv_Mz": 3490.913606285102,
   "mmmm_Mz": 3467.720552289205,
   "dMz_rel": 0.006688270766393299,
   "hdiv_demag": 0.33306637300943714,
   "hdiv_iters": 23,
 